# 1. Setup & Data Ingestion

This notebook handles the complete setup process including:

1. **Environment Setup** - Install dependencies and configure workspace
2. **Unity Catalog Setup** - Create catalogs, schemas, and volumes  
3. **Data Ingestion** - Download and organize source documents
4. **Table Creation** - Set up Delta tables for processed documents

This combines the previous ingestion workflow with modern setup practices.

In [0]:
%pip install uv
%sh uv sync --extra all
%restart_python

In [0]:
# Configuration and imports
import pandas as pd
from pathlib import Path
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import VolumeType

# Configuration
CATALOG = "shm"
SCHEMA = "multimodal" 
RAW_DOCS_VOL = "raw_docs"
PROCESSED_DOCS_VOL = "processed_docs"

print(f"Setting up workspace with:")
print(f"  Catalog: {CATALOG}")
print(f"  Schema: {SCHEMA}")
print(f"  Raw docs volume: {RAW_DOCS_VOL}")
print(f"  Processed docs volume: {PROCESSED_DOCS_VOL}")

In [0]:
# Create Unity Catalog structure
w = WorkspaceClient()

print("🏗️  Creating Unity Catalog structure...")

# Create catalog
try:
    w.catalogs.create(name=CATALOG)
    print(f"✅ Created catalog: {CATALOG}")
except Exception as e:
    print(f"📋 Catalog {CATALOG} already exists")

# Create schema  
try:
    w.schemas.create(catalog_name=CATALOG, name=SCHEMA)
    print(f"✅ Created schema: {SCHEMA}")
except Exception as e:
    print(f"📋 Schema {SCHEMA} already exists")

# Create volumes
for vol_name in [RAW_DOCS_VOL, PROCESSED_DOCS_VOL]:
    try:
        w.volumes.create(
            catalog_name=CATALOG,
            schema_name=SCHEMA,
            name=vol_name,
            volume_type=VolumeType.MANAGED,
        )
        print(f"✅ Created volume: {vol_name}")
    except Exception as e:
        print(f"📋 Volume {vol_name} already exists")

print("\n🎉 Unity Catalog structure is ready!")

## Sample Data Ingestion

For demonstration, we'll ingest some sample documents. In production, this could be replaced with your specific ingestion pipeline.

In [0]:
import pandas as pd
doc_df = pd.read_csv('assets/forge_reports.csv')
doc_downloads = doc_df.download_link.to_list()

In [0]:
# Sample document ingestion
import requests
import os
from pathlib import Path

def sanitize_filename(url):
    """Convert URL to safe filename."""
    filename = url.split('/')[-1]
    if not filename or '.' not in filename:
        filename = url.replace('/', '_').replace(':', '_') + '.pdf'
    return filename

def download_file(url, save_dir):
    """Download file from URL."""
    try:
        filename = sanitize_filename(url)
        filepath = Path(save_dir) / filename
        
        print(f"Downloading: {filename}")
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        with open(filepath, 'wb') as f:
            f.write(response.content)
            
        return str(filepath)
    except Exception as e:
        print(f"Failed to download {url}: {e}")
        return None

# Sample document URLs (replace with your documents)

RAW_DOC_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/{RAW_DOCS_VOL}"

print(f"📥 Downloading sample documents to: {RAW_DOC_DIR}")

downloaded_files = []
for url in doc_downloads:
    filepath = download_file(url, RAW_DOC_DIR)
    if filepath:
        downloaded_files.append(filepath)

print(f"✅ Downloaded {len(downloaded_files)} files")
for file in downloaded_files:
    print(f"  📄 {file}")

## Setup Complete! 

Your workspace is now ready with:

✅ **Unity Catalog Structure**
- Catalog: `main`
- Schema: `default`
- Volumes: `raw_docs`, `processed_docs`

✅ **Document Tables**
- `processed_documents` - Document metadata
- `document_chunks` - Searchable content chunks

✅ **Sample Documents** - Ready for processing

### Next Steps:
1. Run **`2_parse.ipynb`** to process documents with AI_PARSE or Docling
2. Run **`3_parse_ray.ipynb`** for parallel processing with Ray
3. Run **`4_deploy.ipynb`** to deploy serving endpoints
4. Run **`5_evaluate.ipynb`** to evaluate processing resultssure t